In [ ]:
# Step 1: Understand the Signal First

# Before writing any filter code, ask:

# 1. What is the sampling frequency (fs)?

# Example:

fs = 700   # EMG
fs = 4     # EDA
fs = 64    # ACC
fs = 700   # ECG

2. What noise exists?

Common noises:
| Noise Type           | Solution         |
| -------------------- | ---------------- |
| High-frequency noise | Low-pass filter  |
| Baseline drift       | High-pass filter |
| Both                 | Band-pass filter |
| Powerline (50 Hz)    | Notch filter     |
| Random spikes        | Median filter    |


Step 2: Create Reusable Filter Functions

These work for nearly every dataset.

In [ ]:
# low-Pass Filter

# Removes high-frequency noise.

from scipy.signal import butter, filtfilt

def lowpass_filter(signal, cutoff, fs, order=4):
    nyquist = fs / 2
    normal_cutoff = cutoff / nyquist

    b, a = butter(order, normal_cutoff, btype='low')
    filtered = filtfilt(b, a, signal)

    return filtered

# Usage:

filtered = lowpass_filter(raw_signal,
                          cutoff=5,
                          fs=4)

In [ ]:
# High-Pass Filter

# Removes baseline drift.

def highpass_filter(signal, cutoff, fs, order=4):
    nyquist = fs / 2
    normal_cutoff = cutoff / nyquist

    b, a = butter(order, normal_cutoff, btype='high')
    filtered = filtfilt(b, a, signal)

    return filtered

# Usage:

filtered = highpass_filter(raw_signal,
                           cutoff=0.05,
                           fs=4)

In [ ]:
# Band-Pass Filter

# Keeps only useful frequencies.

def bandpass_filter(signal,
                    lowcut,
                    highcut,
                    fs,
                    order=4):

    nyquist = fs / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order,
                  [low, high],
                  btype='band')

    filtered = filtfilt(b, a, signal)

    return filtered

# Usage:

filtered = bandpass_filter(
                signal=raw_signal,
                lowcut=20,
                highcut=450,
                fs=700)

In [ ]:
# Notch Filter

# Removes powerline interference.

from scipy.signal import iirnotch

def notch_filter(signal, freq, fs, Q=30):

    b, a = iirnotch(freq, Q, fs)

    filtered = filtfilt(b, a, signal)

    return filtered

# Usage:

filtered = notch_filter(raw_signal,
                        freq=50,
                        fs=700)

Step 3: Build a Universal Filtering Pipeline

This is how professionals do it.

In [ ]:
def filter_signal(signal,
                  fs,
                  lowcut=None,
                  highcut=None,
                  notch=None):

    filtered = signal.copy()

    # Notch
    if notch:
        filtered = notch_filter(filtered,
                                notch,
                                fs)

    # Bandpass
    if lowcut and highcut:
        filtered = bandpass_filter(
                        filtered,
                        lowcut,
                        highcut,
                        fs)

    # Highpass only
    elif lowcut:
        filtered = highpass_filter(
                        filtered,
                        lowcut,
                        fs)

    # Lowpass only
    elif highcut:
        filtered = lowpass_filter(
                        filtered,
                        highcut,
                        fs)
# usage
    return filtered
filtered_emg = filter_signal(
                    emg,
                    fs=700,
                    lowcut=20,
                    highcut=450,
                    notch=50)